In [ ]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

In [ ]:
# Setup colab
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted")
import os
os.chdir('/content/drive/MyDrive/reranking_rag_and_qw/notebook')
print(f"✅ Working directory: {os.getcwd()}")
print("Contents:", os.listdir("."))

In [ ]:
# Install requirements
#%pip install -r ../requirements.txt
%pip install -q bitsandbytes accelerate

In [ ]:
%pip install -q "sentence-transformers>=3.0"
%pip install -q "jsonlines"
%pip install -q "rank_bm25"
%pip install -q "rouge_score"
%pip install -q "faiss-cpu"

# 3. RankRAG Implementation
Triển khai Cross-encoder và LLM-based reranking

In [ ]:
import sys, os
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

In [ ]:
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from src.reranking.rank_rag import RankReranker, ContextSelector
from src.retrieval.hybrid_retriever import HybridRetriever
from src.generation.llm_generator import LLMGenerator
from src.data.data_loader import DataLoader

In [ ]:
# Setup components
data_loader = DataLoader("../data")
retriever = HybridRetriever()
llm = LLMGenerator(
    provider="groq",
    groq_api_key=GROQ_API_KEY,
    groq_model="llama-3.1-8b-instant"
)

import pickle
import faiss
import numpy as np

print("Loading cached index...")

with open("cache/corpus.pkl", "rb") as f:
    retriever.corpus = pickle.load(f)

with open("cache/bm25.pkl", "rb") as f:
    retriever.bm25 = pickle.load(f)

retriever.faiss_index = faiss.read_index("cache/faiss.index")

print(f"Loaded index with {len(retriever.corpus)} documents")
print(f"Sample: {retriever.corpus[0][:100]}")

print(f"Index ready with {len(retriever.corpus)} valid documents")
print(f"Sample doc: {retriever.corpus[0][:100] if retriever.corpus else 'EMPTY!'}")

In [ ]:
print(dir(data_loader))
# If there's a method like list_datasets() or get_dataset_names(), call it:
# print(data_loader.list_datasets())
# If it's a direct attribute, access it:
# print(data_loader.dataset_names)

# Also check the directory where data is loaded from
import os
print("Contents of ../data:")
print(os.listdir('../data'))

## Cross-Encoder Reranking

In [ ]:
import json

try:
    with open("../models/model_info.json", "r", encoding="utf-8") as f:
        model_info = json.load(f)
    reranker_model_path = model_info["cross_encoder_model"]
    print(f"Loading Fine-tuned model: {reranker_model_path}")
except FileNotFoundError:
    reranker_model_path = "BAAI/bge-reranker-v2-m3"
    print(f"Loading Default model: {reranker_model_path}")

cross_reranker = RankReranker(
    method="cross-encoder",
    cross_encoder_model=reranker_model_path,
    ce_weight=1.0,
    llm_weight=0.0
)

# Retrieve documents
query = "Triệu chứng bệnh đái tháo đường là gì?"
raw_docs = retriever.retrieve([query], top_k=10)

print(f"Retrieved {len(raw_docs)} documents")
print("\nBefore Reranking (Top 5):")
for i, doc in enumerate(raw_docs[:5], 1):
    print(f"{i}. [Score: {doc['score']:.3f}] {doc['text'][:100]}...")

In [ ]:
#Rerank
reranked_docs = cross_reranker.rerank(query, raw_docs, top_k=5)

#Content selector
selector = ContextSelector(top_k=5, deduplicate=True, max_tokens=300)
selected_docs = selector.select(reranked_docs)

print("\nAfter Reranking + Context Selection:")
for i, doc in enumerate(selected_docs, 1):
    score = doc.get("reranker_score", 0)
    print(f"{i}. [Score: {score:.3f}] {doc['text'][:100]}...")

## LLM-based Reranking

In [ ]:
# Initialize LLM reranker
llm_reranker = RankReranker(method="llm", llm_client=llm)

# Rerank with LLM
llm_reranked = llm_reranker.rerank(query, raw_docs, top_k=5)

print("\nLLM-based Reranking (Top 5):")
for i, doc in enumerate(llm_reranked, 1):
    score = doc.get("reranker_score", 0)
    print(f"{i}. [Score: {score:.3f}] {doc['text'][:100]}...")

In [ ]:
print(llm_reranker.method, llm_reranker.cross_encoder)

## Hybrid Reranking Comparison

In [ ]:
# Compare different reranking methods
test_queries = [
    "Bệnh huyết áp cao",
    "Cách phòng ngừa COVID-19",
    "Triệu chứng bệnh tim mạch"
]

comparison_results = {}

for test_query in test_queries:
    raw_docs = retriever.retrieve([test_query], top_k=10)

    # Cross-encoder
    ce_reranked_raw = cross_reranker.rerank(test_query, raw_docs.copy(), top_k=10)
    ce_reranked = ContextSelector(top_k=3, deduplicate=True).select(ce_reranked_raw)

    # No reranking (baseline)
    no_rerank = raw_docs[:3]

    comparison_results[test_query] = {
        "no_rerank": no_rerank,
        "cross_encoder": ce_reranked
    }

# Display comparison
for query, results in comparison_results.items():
    print(f"\nQuery: {query}")
    print("\n  Baseline (No Reranking):")
    for i, doc in enumerate(results["no_rerank"], 1):
        print(f"    {i}. {doc['text'][:80]}...")
    print("\n  With Cross-Encoder Reranking:")
    for i, doc in enumerate(results["cross_encoder"], 1):
        print(f"    {i}. {doc['text'][:80]}...")

## Thực nghiệm 1: Các phương pháp Reranking
- LLM-based Reranking: Dùng LLM thay cross-encoder để chấm điểm)
- Hybrid Reranking Comparison: So sánh baseline (không rerank) vs cross-encoder trên nhiều query, kiểm tra reranker có thực sự cải thiện thứ tự không
- Evaluate on Test Set: đo định lượng (F1, relevance) trên 10 mẫu thật, trừ tập **uit_viquad2**

In [ ]:
from tqdm import tqdm
from IPython.display import display
import pandas as pd
from evaluation.evaluator import RAGEvaluator
import copy

evaluator = RAGEvaluator()
dataset_names = ["vimpa"]

all_dataset_results = {}

for dataset_name in dataset_names:
    print(f"\nEvaluating dataset: {dataset_name}")
    test_data = data_loader.load_dataset(dataset_name, "test")[:20]

    metrics_without = {"f1": 0, "relevance": 0, "similarity": 0, "faithfulness": 0}
    metrics_with    = {"f1": 0, "relevance": 0, "similarity": 0, "faithfulness": 0}
    count = 0
    debug_count = 0

    for sample in tqdm(test_data, desc=f"Evaluating {dataset_name}"):
        query        = sample.get("query", "")
        ground_truth = sample.get("ground_truth", "")
        if not query or not ground_truth:
            continue

        docs_raw     = retriever.retrieve([query], top_k=10)
        docs_without = docs_raw[:5]
        docs_with    = ContextSelector(top_k=5, deduplicate=True).select(
                           cross_reranker.rerank(query, copy.deepcopy(docs_raw), top_k=10))

        answer_without = llm.generate(query, docs_without, max_tokens=300)
        answer_with    = llm.generate(query, docs_with,    max_tokens=300)

        m_wo = evaluator.calculate_all(query, answer_without, docs_without, ground_truth)
        m_wi = evaluator.calculate_all(query, answer_with,    docs_with,    ground_truth)

        for k in metrics_without:
            metrics_without[k] += m_wo.get(k, 0)
            metrics_with[k]    += m_wi.get(k, 0)
        count += 1

    rows = []
    for k in metrics_without:
        wo  = metrics_without[k] / count if count else 0
        wi  = metrics_with[k]    / count if count else 0
        imp = (wi - wo) / wo * 100 if wo else 0
        rows.append({"Metric": k.upper(), "Without Reranking": f"{wo:.4f}",
                     "With Reranking": f"{wi:.4f}", "Improvement": f"{imp:+.2f}%"})

    all_dataset_results[dataset_name] = pd.DataFrame(rows).set_index("Metric")

# Display all results
print("\n===== Aggregated Evaluation Results =====")
for dataset_name, df_result in all_dataset_results.items():
    print(f"\nResults for {dataset_name}:")
    display(df_result)